# Estrazione evidenza dal DWH (`dw_ram_products`)

Estrae dalla collection consolidata su MongoDB Atlas:
- conteggio totale documenti + breakdown per `market`/`ram_category` (verifica incrociata con i numeri dichiarati in slide 16)
- un documento di esempio pulito, da usare come evidenza visiva nella slide

**Richiede**: un file `.env` nella stessa cartella con `MONGO_URI=mongodb+srv://...`


In [14]:
import json
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = "idealo_ram"
COLLECTION_NAME = "dw_ram_products"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
coll = db[COLLECTION_NAME]

## Conteggio totale e breakdown per market / ram_category

In [15]:
total = coll.count_documents({})
print(f"Totale documenti in {COLLECTION_NAME}: {total}\n")

print("Breakdown per market / ram_category:")
pipeline = [
    {"$group": {"_id": {"market": "$market", "ram_category": "$ram_category"}, "count": {"$sum": 1}}},
    {"$sort": {"_id.market": 1, "_id.ram_category": 1}},
]
for row in coll.aggregate(pipeline):
    print(f"  {row['_id']['market']} - {row['_id']['ram_category']}: {row['count']}")

Totale documenti in dw_ram_products: 656

Breakdown per market / ram_category:
  DE - DDR4: 174
  DE - DDR5: 156
  IT - DDR4: 174
  IT - DDR5: 152


## Documento di esempio

Filtro per un documento IT/DDR4 con `capacity_gb` e `frequency_mts` valorizzati,
per evitare di pescare per caso un record con campi mancanti come primo esempio.

In [25]:
sample = coll.find_one({
    "market": "IT",
    "ram_category": "DDR4",
    "capacity_gb": {"$ne": None},
    "frequency_mts": {"$ne": None},
    "offerId": {"$eq": "21f5a8d1a3cb355ed1e1e6271c68252b"},
})

if sample is None:
    # fallback: un documento qualsiasi, se il filtro sopra non trova nulla
    sample = coll.find_one({})

# Rimuovi l'_id di MongoDB (ObjectId non è serializzabile in JSON puro e non serve mostrarlo)
sample.pop("_id", None)

print(json.dumps(sample, ensure_ascii=False, indent=2))

{
  "id": "200174645",
  "offerId": "44b5a6aeee2d6bde5199433d1467c14a",
  "title": "Kingston ValueRam 16GB DDR4-3200 CL22 (KVR32N22D8/16)",
  "subheading": "RAM DDR4",
  "userReviewCount": null,
  "roundedRating": null,
  "rawPrice": 120.94,
  "offerCount": 17,
  "usedOnly": false,
  "shopName": "PC Componentes IT",
  "shippingCosts": 0.0,
  "shippingIsFreeOfCharge": false,
  "isBestseller": false,
  "total_capacity_gb": 16.0,
  "module_capacity_gb": 16.0,
  "num_modules": 1.0,
  "frequency_mts": 3200.0,
  "cas_latency_raw": "CL 22-22-22",
  "cas_latency_primary": 22.0,
  "voltage_v": 1.2,
  "form_factor": "UDIMM",
  "market": "IT",
  "ram_category": "DDR4"
}


## Salvataggio su file

Utile per ricaricare l'esempio e trasformarlo in immagine per la slide.

In [17]:
with open("dwh_sample_document.json", "w", encoding="utf-8") as f:
    json.dump(sample, f, ensure_ascii=False, indent=2)

with open("dwh_counts.json", "w", encoding="utf-8") as f:
    counts = {"total": total}
    json.dump(counts, f, ensure_ascii=False, indent=2)

print("Salvati: dwh_sample_document.json, dwh_counts.json")

Salvati: dwh_sample_document.json, dwh_counts.json
